In [2]:
import pandas as pd

# reload the Phase 2 master tables (in case kernel was restarted)
hyp_table = pd.read_csv(r"C:\tri_disease_project\data\processed\hypertension_master.csv")
obesity_table = pd.read_csv(r"C:\tri_disease_project\data\processed\obesity_master.csv")
t2d_table = pd.read_csv(r"C:\tri_disease_project\data\processed\t2d_master.csv")

# apply significance thresholds BEFORE building gene sets
GWAS_PVAL_THRESHOLD = 5e-8
PHEGENI_PVAL_THRESHOLD = 5e-8
OT_SCORE_THRESHOLD = 0.1

def filter_disease_table(df):
    sig_gwas = df['gwas_pvalue'].notna() & (df['gwas_pvalue'] < GWAS_PVAL_THRESHOLD)
    sig_phegeni = df['phegeni_pvalue'].notna() & (df['phegeni_pvalue'] < PHEGENI_PVAL_THRESHOLD)
    sig_ot = df['ot_score'].notna() & (df['ot_score'] >= OT_SCORE_THRESHOLD)
    keep = sig_gwas | sig_phegeni | sig_ot
    return df.loc[keep].copy()

hyp_table_filt = filter_disease_table(hyp_table)
obesity_table_filt = filter_disease_table(obesity_table)
t2d_table_filt = filter_disease_table(t2d_table)

print("Hypertension: before =", len(hyp_table), "-> after significance filter =", len(hyp_table_filt))
print("Obesity: before =", len(obesity_table), "-> after significance filter =", len(obesity_table_filt))
print("T2D: before =", len(t2d_table), "-> after significance filter =", len(t2d_table_filt))

hyp_genes = set(hyp_table_filt['hgnc_symbol'])
obesity_genes = set(obesity_table_filt['hgnc_symbol'])
t2d_genes = set(t2d_table_filt['hgnc_symbol'])

shared_all3 = hyp_genes & obesity_genes & t2d_genes
hyp_obesity = hyp_genes & obesity_genes
hyp_t2d = hyp_genes & t2d_genes
obesity_t2d = obesity_genes & t2d_genes

print("\nHypertension gene set size:", len(hyp_genes))
print("Obesity gene set size:", len(obesity_genes))
print("T2D gene set size:", len(t2d_genes))

print("\nShared: Hypertension ∩ Obesity ∩ T2D:", len(shared_all3))
print("Shared: Hypertension ∩ Obesity:", len(hyp_obesity))
print("Shared: Hypertension ∩ T2D:", len(hyp_t2d))
print("Shared: Obesity ∩ T2D:", len(obesity_t2d))

print("\nTri-disease shared genes:")
print(sorted(shared_all3))

import os
os.makedirs(r"C:\tri_disease_project\data\processed", exist_ok=True)
hyp_table_filt.to_csv(r"C:\tri_disease_project\data\processed\hypertension_master_filtered.csv", index=False)
obesity_table_filt.to_csv(r"C:\tri_disease_project\data\processed\obesity_master_filtered.csv", index=False)
t2d_table_filt.to_csv(r"C:\tri_disease_project\data\processed\t2d_master_filtered.csv", index=False)

print("\nSaved filtered master tables to data/processed/")

Hypertension: before = 9306 -> after significance filter = 4500
Obesity: before = 13915 -> after significance filter = 7464
T2D: before = 10688 -> after significance filter = 3755

Hypertension gene set size: 4500
Obesity gene set size: 7464
T2D gene set size: 3755

Shared: Hypertension ∩ Obesity ∩ T2D: 908
Shared: Hypertension ∩ Obesity: 2339
Shared: Hypertension ∩ T2D: 1220
Shared: Obesity ∩ T2D: 2155

Tri-disease shared genes:
['ABCC8', 'ABHD5', 'ABO', 'ACAN', 'ACE', 'ACE2', 'ACHE', 'ACOXL', 'ACTG1P22', 'ACVR1C', 'ACVR2A', 'ADAMTS9', 'ADAMTS9-AS2', 'ADAMTSL3', 'ADAP2', 'ADARB2', 'ADCY5', 'ADCY9', 'ADGRV1', 'ADH1B', 'ADH4', 'ADIPOQ', 'ADO', 'ADRA1D', 'ADRA2A', 'ADRA2B', 'ADRA2C', 'ADRB1', 'ADRB2', 'AFF1', 'AGBL2', 'AGBL4', 'AGER', 'AGMO', 'AGTR1', 'AIDAP3', 'AK1', 'AK6P1', 'AKNA', 'AKR1B1', 'ALDH1A2', 'ALDH2', 'AMZ1', 'ANAPC4', 'ANGPTL4', 'ANKRD31', 'AOC1', 'AOPEP', 'APOA1', 'APOA5', 'APOB', 'APOC1', 'APOE', 'APOH', 'ARHGAP24', 'ARHGAP42', 'ARID4A', 'ARID5B', 'ARL14EP', 'ARL15', 'ARM

In [3]:
import mygene
mg = mygene.MyGeneInfo()

shared_list = sorted(shared_all3)
gene_type_result = mg.querymany(shared_list, scopes='symbol', fields='type_of_gene',
                                 species='human', as_dataframe=True)
gene_type_result = gene_type_result.reset_index().rename(columns={'query': 'hgnc_symbol'})
gene_type_result = gene_type_result.drop_duplicates(subset='hgnc_symbol', keep='first')

print("Gene type breakdown among the 908 shared genes:")
print(gene_type_result['type_of_gene'].value_counts())

protein_coding_genes = set(gene_type_result.loc[gene_type_result['type_of_gene'] == 'protein-coding', 'hgnc_symbol'])
dropped_genes = shared_all3 - protein_coding_genes

print("\nProtein-coding shared genes:", len(protein_coding_genes))
print("Dropped (non-coding/pseudogene/ncRNA/unresolved):", len(dropped_genes))

# this becomes our final Phase 3 shared gene set going forward
shared_all3_coding = protein_coding_genes

print("\nFinal protein-coding tri-disease shared gene list:")
print(sorted(shared_all3_coding))

import pandas as pd
pd.Series(sorted(shared_all3_coding), name='hgnc_symbol').to_csv(
    r"C:\tri_disease_project\data\processed\shared_genes_protein_coding.csv", index=False)

70 input query terms found dup hits:	[('AIDAP3', 2), ('ARMC10P1', 2), ('ATP5PBP6', 2), ('C20orf181', 2), ('C5orf67', 2), ('CATSPER2P1', 2


Gene type breakdown among the 908 shared genes:
type_of_gene
protein-coding    690
ncRNA              92
pseudo             55
unknown             2
snRNA               1
snoRNA              1
Name: count, dtype: int64

Protein-coding shared genes: 690
Dropped (non-coding/pseudogene/ncRNA/unresolved): 218

Final protein-coding tri-disease shared gene list:
['ABCC8', 'ABHD5', 'ABO', 'ACAN', 'ACE', 'ACE2', 'ACHE', 'ACOXL', 'ACVR1C', 'ACVR2A', 'ADAMTS9', 'ADAMTSL3', 'ADAP2', 'ADARB2', 'ADCY5', 'ADCY9', 'ADGRV1', 'ADH1B', 'ADH4', 'ADIPOQ', 'ADO', 'ADRA1D', 'ADRA2A', 'ADRA2B', 'ADRA2C', 'ADRB1', 'ADRB2', 'AFF1', 'AGBL2', 'AGBL4', 'AGER', 'AGMO', 'AGTR1', 'AK1', 'AKNA', 'AKR1B1', 'ALDH1A2', 'ALDH2', 'AMZ1', 'ANAPC4', 'ANGPTL4', 'ANKRD31', 'AOC1', 'AOPEP', 'APOA1', 'APOA5', 'APOB', 'APOC1', 'APOE', 'APOH', 'ARHGAP24', 'ARHGAP42', 'ARID4A', 'ARID5B', 'ARL14EP', 'ARL15', 'ARPP21', 'ASAH1', 'ASB3', 'ASCC2', 'ASTN2', 'ATG7', 'ATP2B4', 'ATP5F1E', 'ATXN2', 'AXIN1', 'BANP', 'BCKDHB', 'BCL11B', 'BCL2

In [4]:
from scipy.stats import hypergeom
from statsmodels.stats.multitest import multipletests

N = 20000  # background: total protein-coding genes in human genome

def pairwise_hypergeom_test(set_a, set_b, N):
    K = len(set_a)
    n = len(set_b)
    overlap = len(set_a & set_b)
    pval = hypergeom.sf(overlap - 1, N, K, n)
    return overlap, pval

tests = []
overlap, p = pairwise_hypergeom_test(hyp_genes, obesity_genes, N)
tests.append(("Hypertension vs Obesity", len(hyp_genes), len(obesity_genes), overlap, p))

overlap, p = pairwise_hypergeom_test(hyp_genes, t2d_genes, N)
tests.append(("Hypertension vs T2D", len(hyp_genes), len(t2d_genes), overlap, p))

overlap, p = pairwise_hypergeom_test(obesity_genes, t2d_genes, N)
tests.append(("Obesity vs T2D", len(obesity_genes), len(t2d_genes), overlap, p))

# tri-disease: test (Hyp ∩ Obesity) vs T2D, using the PROTEIN-CODING shared count for the actual overlap
K3 = len(hyp_obesity)
n3 = len(t2d_genes)
overlap3 = len(shared_all3_coding)  # use the cleaned, protein-coding overlap
pval3 = hypergeom.sf(overlap3 - 1, N, K3, n3)
tests.append(("(Hyp ∩ Obesity) vs T2D [tri-disease, protein-coding]", K3, n3, overlap3, pval3))

results_df = pd.DataFrame(tests, columns=["Comparison", "Set_A_size", "Set_B_size", "Overlap", "p_value"])

reject, pvals_corrected, _, _ = multipletests(results_df['p_value'], method='fdr_bh')
results_df['p_adj_fdr'] = pvals_corrected
results_df['significant_fdr_0.05'] = reject

print(results_df.to_string(index=False))

results_df.to_csv(r"C:\tri_disease_project\data\processed\phase3_hypergeometric_results.csv", index=False)

                                          Comparison  Set_A_size  Set_B_size  Overlap       p_value     p_adj_fdr  significant_fdr_0.05
                             Hypertension vs Obesity        4500        7464     2339 2.387005e-115 4.774011e-115                  True
                                 Hypertension vs T2D        4500        3755     1220  5.280998e-56  7.041330e-56                  True
                                      Obesity vs T2D        7464        3755     2155 5.434398e-170 2.173759e-169                  True
(Hyp ∩ Obesity) vs T2D [tri-disease, protein-coding]        2339        3755      690  2.691584e-41  2.691584e-41                  True


In [5]:
import numpy as np

# pull each shared gene's evidence from all three disease tables (using the significance-filtered versions)
def get_evidence(gene, table):
    row = table[table['hgnc_symbol'] == gene]
    if row.empty:
        return np.nan, np.nan, np.nan
    return row['gwas_pvalue'].values[0], row['phegeni_pvalue'].values[0], row['ot_score'].values[0]

records = []
for gene in sorted(shared_all3_coding):
    hyp_gwas, hyp_phe, hyp_ot = get_evidence(gene, hyp_table_filt)
    ob_gwas, ob_phe, ob_ot = get_evidence(gene, obesity_table_filt)
    t2d_gwas, t2d_phe, t2d_ot = get_evidence(gene, t2d_table_filt)
    records.append({
        'hgnc_symbol': gene,
        'hyp_gwas_p': hyp_gwas, 'hyp_phegeni_p': hyp_phe, 'hyp_ot_score': hyp_ot,
        'obesity_gwas_p': ob_gwas, 'obesity_phegeni_p': ob_phe, 'obesity_ot_score': ob_ot,
        't2d_gwas_p': t2d_gwas, 't2d_phegeni_p': t2d_phe, 't2d_ot_score': t2d_ot,
    })

evidence_df = pd.DataFrame(records)

# combined evidence score: min-max normalize each column, then average
# for p-values: use -log10(p) so smaller p-values -> larger (better) scores, then normalize
def normalize_pval_cols(df, cols):
    for c in cols:
        neglog = -np.log10(df[c].replace(0, 1e-300))  # avoid log(0)
        df[c + '_norm'] = (neglog - neglog.min()) / (neglog.max() - neglog.min())
    return df

def normalize_score_cols(df, cols):
    for c in cols:
        df[c + '_norm'] = (df[c] - df[c].min()) / (df[c].max() - df[c].min())
    return df

pval_cols = ['hyp_gwas_p', 'hyp_phegeni_p', 'obesity_gwas_p', 'obesity_phegeni_p', 't2d_gwas_p', 't2d_phegeni_p']
score_cols = ['hyp_ot_score', 'obesity_ot_score', 't2d_ot_score']

evidence_df = normalize_pval_cols(evidence_df, pval_cols)
evidence_df = normalize_score_cols(evidence_df, score_cols)

norm_cols = [c + '_norm' for c in pval_cols + score_cols]
evidence_df['combined_evidence_score'] = evidence_df[norm_cols].mean(axis=1, skipna=True)

# rank
evidence_df = evidence_df.sort_values('combined_evidence_score', ascending=False).reset_index(drop=True)
evidence_df.insert(0, 'rank', evidence_df.index + 1)

print("Top 30 ranked shared genes by combined evidence strength:")
print(evidence_df[['rank', 'hgnc_symbol', 'combined_evidence_score']].head(30).to_string(index=False))

evidence_df.to_csv(r"C:\tri_disease_project\data\processed\shared_genes_ranked.csv", index=False)
print("\nSaved full ranked table to data/processed/shared_genes_ranked.csv")

Top 30 ranked shared genes by combined evidence strength:
 rank hgnc_symbol  combined_evidence_score
    1       AGTR1                 0.695489
    2      SLC5A2                 0.603578
    3         FTO                 0.594420
    4       GLP1R                 0.518668
    5       NR3C2                 0.507686
    6      TCF7L2                 0.439613
    7      ADRA2B                 0.430419
    8       RSPO3                 0.421134
    9      MAP4K5                 0.394714
   10      KCNJ11                 0.389998
   11       PPARG                 0.375776
   12        GIPR                 0.373995
   13       PDE3B                 0.372208
   14       PTGS2                 0.368652
   15         LPL                 0.368621
   16       PDE4A                 0.364377
   17      CDKAL1                 0.362609
   18      ADRA1D                 0.357967
   19       KCNJ8                 0.355458
   20       ADRB2                 0.353735
   21      VPS37D                 0.351